# burnscars thin Colab runner

Zero logic in cells: every cell calls `burnscars.colab` or `tools/*.py`.

**Colab compute is shared with other projects. Do not run a non-smoke cell unless the allocation id is marked APPROVED in `docs/COMPUTE_LEDGER.md`.** GPU-tier tools refuse to start otherwise.

Secrets: `GH_TOKEN` (and later `HF_TOKEN` if ever needed) live in the Colab secrets panel only.

In [ ]:
# 1. Mount Drive (durable storage; never the working directory)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Auth BEFORE clone (token from the secrets panel, never pasted)
import os
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

In [ ]:
# 3. Clone or update. Re-run this cell after pushing a fix; same kernel.
!test -d /content/Burn-Scars || git clone https://$GH_TOKEN@github.com/meteorboyF/Burn-Scars.git /content/Burn-Scars
!cd /content/Burn-Scars && git fetch -q origin && git reset -q --hard origin/main

In [ ]:
# 4. Environment: local caches/outputs, fingerprint-gated install, printed SHA
import sys; sys.path.insert(0, '/content/Burn-Scars/src')
import importlib, burnscars.colab as bc; importlib.reload(bc)
bc.configure_env(); bc.ensure_requirements(); bc.print_sha()

In [ ]:
# 5. Allocation id approved in docs/COMPUTE_LEDGER.md (leave None for smoke only)
os.environ['BURNSCARS_ALLOCATION'] = ''

## Steps (one `run()` per cell)

In [ ]:
bc.run('plumbing_check', config='plumbing_check', smoke=True)

In [ ]:
# copy outputs to Drive after each step
bc.copy_to_drive('plumbing_check')